# 🚀 CineGrade AI - Free GPU Backend

Welcome! This notebook provides the 100% free NVIDIA T4 GPU backend required for CineGrade AI.

### Instructions (Quick Start Guide):
1. In the menu at the top, click **Runtime** ➔ **Run all** (or press Ctrl+F9).
2. A popup might warn you that this notebook was not authored by Google. Click **Run anyway**.
3. Scroll down to the bottom cell. It will print out a **Cloudflare Tunnel URL** (it looks like `https://xyz.trycloudflare.com`).
4. Copy that URL and paste it into the CineGrade web app.

In [ ]:
# @title 1. Install Dependencies
import os

if not os.path.exists("VideoColorGrading"):
    !git clone https://github.com/shameel0505/VideoColorGrading.git
%cd VideoColorGrading

print("⏳ Installing required packages & Cloudflared...")
!pip install -q fastapi uvicorn pillow pillow-lut imageio imageio-ffmpeg rawpy python-multipart
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb

print("\n✅ Setup complete!")

In [ ]:
# @title 2. Start GPU Server & Cloudflare Tunnel
import subprocess
import time
import os
import urllib.request
import re

print('⏳ Cleaning up previous server & tunnel processes...')
subprocess.run('pkill -f uvicorn', shell=True)
subprocess.run('pkill -f cloudflared', shell=True)
time.sleep(1)

print('⏳ Booting up FastAPI Server on port 8444...')
log_file = open('uvicorn.log', 'w')
fastapi_proc = subprocess.Popen(
    ['uvicorn', 'api:app', '--host', '127.0.0.1', '--port', '8444'],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

server_up = False
for i in range(45):
    time.sleep(1)
    if fastapi_proc.poll() is not None:
        print('\n❌ ERROR: FastAPI crashed! uvicorn.log contents:\n')
        with open('uvicorn.log', 'r') as f:
            print(f.read())
        break
    try:
        urllib.request.urlopen('http://127.0.0.1:8444/api/library')
        server_up = True
        break
    except Exception:
        pass

if not server_up:
    print('❌ Failed to start local FastAPI server.')
else:
    print('✅ FastAPI Server is UP and healthy! Establishing Cloudflare Edge Tunnel...')

    cf_log = open('cf.log', 'w')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8444'],
        stdout=cf_log,
        stderr=subprocess.STDOUT
    )

    url_found = False
    for attempt in range(30):
        time.sleep(1)
        if os.path.exists('cf.log'):
            with open('cf.log', 'r') as f:
                content = f.read()
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                if match:
                    full_url = match.group(0)
                    print('\n' + '⭐'*50)
                    print('COPY THIS CLOUDFLARE URL FOR THE WEB APP:')
                    print(f'\n--->  {full_url}  <---\n')
                    print('⭐'*50 + '\n')
                    url_found = True
                    break

    if not url_found:
        print('❌ Failed to generate Cloudflare URL. Check cf.log:')
        with open('cf.log', 'r') as f:
            print(f.read())
    else:
        try:
            tunnel_proc.wait()
        except KeyboardInterrupt:
            fastapi_proc.terminate()
            print('\nServer stopped.')
